# BANKING77 Intent Classification — Fine-Tuning DistilBERT on Colab T4 GPU

This self-contained notebook fine-tunes `distilbert-base-uncased` on the exact same `mteb/banking77` dataset, canonical 77-class label mapping, and stratified 85/15 train/validation split (`SEED = 42`) used in the local pipeline.

### Steps to Run
1. In Google Colab, select **Runtime -> Change runtime type -> T4 GPU**.
2. Click **Runtime -> Run all** (~3.5–4.5 minutes total with early stopping).
3. When `transformer_intent.zip` downloads, extract its contents directly into:
   `INDV/intent_classifier/models/transformer_intent/` (so `config.json` and `model.safetensors` sit inside that folder).

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device:', torch.cuda.get_device_name(0))
!pip -q install transformers datasets accelerate safetensors scikit-learn

In [ ]:
import json
import math
import random
import re
import unicodedata
import numpy as np
import torch
from datasets import Dataset, load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

SEED = 42
VAL_SIZE = 0.15
DATASET_ID = 'mteb/banking77'
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 96
EPOCHS = 10
BATCH_SIZE = 16
LR = 5e-5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

_CONTROL_RE = re.compile(r'[\u200b-\u200d\ufeff\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')
_WS_RE = re.compile(r'\s+')

def clean_transformer_minimal(text: str) -> str:
    norm = unicodedata.normalize('NFKC', str(text))
    norm = _CONTROL_RE.sub('', norm)
    return _WS_RE.sub(' ', norm).strip()

raw_ds = load_dataset(DATASET_ID)
raw_train_texts = [clean_transformer_minimal(t) for t in raw_ds['train']['text']]
raw_train_str = [str(s) for s in raw_ds['train']['label_text']]
test_texts = [clean_transformer_minimal(t) for t in raw_ds['test']['text']]
test_str = [str(s) for s in raw_ds['test']['label_text']]

label_names = sorted(set(raw_train_str) | set(test_str), key=lambda s: s.lower())
assert len(label_names) == 77
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for i, name in enumerate(label_names)}

raw_train_labels = [label2id[s] for s in raw_train_str]
test_labels = [label2id[s] for s in test_str]

train_texts, val_texts, train_labels, val_labels = train_test_split(
    raw_train_texts,
    raw_train_labels,
    test_size=VAL_SIZE,
    stratify=raw_train_labels,
    random_state=SEED,
)
print(f'Split sizes -> train: {len(train_texts)}, val: {len(val_texts)}, test: {len(test_texts)}')

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_hf_dataset(texts, labels):
    ds = Dataset.from_dict({'text': texts, 'label': labels})
    return ds.map(
        lambda b: tokenizer(b['text'], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=['text'],
    )

train_ds = make_hf_dataset(train_texts, train_labels)
val_ds = make_hf_dataset(val_texts, val_labels)
test_ds = make_hf_dataset(test_texts, test_labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = float(accuracy_score(labels, preds))
    mp, mr, mf1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    wp, wr, wf1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    return {
        'accuracy': acc,
        'macro_precision': float(mp),
        'macro_recall': float(mr),
        'macro_f1': float(mf1),
        'weighted_f1': float(wf1),
    }

In [ ]:
import inspect
from transformers import AutoModelForSequenceClassification, EarlyStoppingCallback, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

total_steps = max(1, math.ceil(len(train_ds) / BATCH_SIZE) * EPOCHS)
warmup_steps = max(1, int(0.06 * total_steps))

ta_kwargs = dict(
    output_dir='distilbert_ckpts',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to='none',
)
ta_sig = inspect.signature(TrainingArguments.__init__).parameters
if 'evaluation_strategy' in ta_sig and 'eval_strategy' not in ta_sig:
    ta_kwargs['evaluation_strategy'] = ta_kwargs.pop('eval_strategy')
training_args = TrainingArguments(**{k: v for k, v in ta_kwargs.items() if k in ta_sig})

tr_sig = inspect.signature(Trainer.__init__).parameters
tr_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
if 'processing_class' in tr_sig:
    tr_kwargs['processing_class'] = tokenizer
else:
    tr_kwargs['tokenizer'] = tokenizer

trainer = Trainer(**tr_kwargs)
train_result = trainer.train()

In [ ]:
import os
import shutil
from pathlib import Path

val_metrics = trainer.evaluate(val_ds)
test_metrics = trainer.evaluate(test_ds)
print('Validation Metrics:', {k: round(v, 4) for k, v in val_metrics.items() if isinstance(v, float)})
print('Test Metrics:      ', {k: round(v, 4) for k, v in test_metrics.items() if isinstance(v, float)})

save_dir = Path('transformer_intent')
save_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

epoch_history = [
    {k: (float(v) if isinstance(v, (int, float)) else v) for k, v in e.items()}
    for e in trainer.state.log_history
    if 'eval_macro_f1' in e
]
meta = {
    'base_model': MODEL_NAME,
    'dataset_source': DATASET_ID,
    'num_labels': len(label_names),
    'max_length': MAX_LENGTH,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LR,
    'seed': SEED,
    'device_trained': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'train_runtime_sec': float(train_result.metrics.get('train_runtime', 0.0)),
    'val_metrics': {k: float(v) for k, v in val_metrics.items() if isinstance(v, (int, float))},
    'test_metrics': {k: float(v) for k, v in test_metrics.items() if isinstance(v, (int, float))},
    'epoch_history': epoch_history,
}
(save_dir / 'training_metadata.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')

print('\nSanity Check Predictions:')
trainer.model.eval()
dev = trainer.model.device
for q in [
    'I lost my debit card and need to freeze it right away.',
    'My contactless tap is not working at checkout',
    'Why is my card payment still pending?',
]:
    enc = tokenizer(q, return_tensors='pt', truncation=True, max_length=MAX_LENGTH)
    enc = {k: v.to(dev) for k, v in enc.items()}
    with torch.no_grad():
        probs = torch.softmax(trainer.model(**enc).logits, dim=-1)[0]
    top_idx = int(torch.argmax(probs))
    print(f'  {q} -> {label_names[top_idx]} ({float(probs[top_idx]):.3f})')

zip_path = shutil.make_archive('transformer_intent', 'zip', root_dir=str(save_dir))
print('Packaged archive:', zip_path, f'({os.path.getsize(zip_path) / 1024**2:.1f} MB)')

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not running inside Google Colab; copy transformer_intent/ locally.')